# P7 — Autonomous Financial Analyst Agent with Tool-Calling
A safe local tool-routing demo: SQL-like portfolio lookup, market-document retrieval and calculator. No external financial advice or live market data.

In [58]:
!pip -q install pandas

In [59]:
from pathlib import Path
import sqlite3
import pandas as pd

BASE_DIR = Path(
    "/kaggle/input/datasets/babblu123/"
    "real-world-ml-genai-kaggle-complete/"
    "real_world_ml_genai_lab"
)

DATA_DIR = BASE_DIR / "datasets"
DOCS_DIR = BASE_DIR / "docs"

# Load market outlook document
market_path = DOCS_DIR / "p7_market_outlook.txt"

print("Docs directory:", DOCS_DIR)
print("Available documents:", [p.name for p in DOCS_DIR.iterdir()])

if not market_path.exists():
    raise FileNotFoundError(
        f"Missing file: {market_path}\n"
        "Check the available documents printed above."
    )

market = market_path.read_text(encoding="utf-8")

print("Market outlook loaded:", len(market), "characters")
print(market[:500])

Docs directory: /kaggle/input/datasets/babblu123/real-world-ml-genai-kaggle-complete/real_world_ml_genai_lab/docs
Available documents: ['p10_logistics_policy.md', 'p5_clinical_guidelines.md', 'p5_sample_note.txt', 'p6_contracts.txt', 'p7_market_outlook.txt']
Market outlook loaded: 353 characters
Q3 Market Outlook (synthetic demonstration only)
The sample outlook assumes elevated volatility, diversified exposure, and periodic portfolio rebalancing. This is not investment advice. Risk assessment should consider customer objectives, time horizon, liquidity needs, tax jurisdiction, and verified current holdings. No return forecast is guaranteed.



In [60]:
def portfolio_lookup(customer_id):
    # Fixed parameterized query; do not accept arbitrary SQL from the user.
    return pd.read_sql_query("SELECT * FROM portfolios WHERE customer_id = ?",conn,params=(int(customer_id),))
def market_search(query):
    words=set(re.findall(r"\w+",query.lower()))
    sentences=re.split(r"(?<=[.!?])\s+",market)
    return sorted(sentences,key=lambda s:len(words & set(re.findall(r"\w+",s.lower()))),reverse=True)[:2]
def safe_calculator(expression):
    # Allow only numeric arithmetic characters; never use eval on arbitrary input.
    if not re.fullmatch(r"[0-9.()+*/ -]+",expression): raise ValueError("Only arithmetic expressions are allowed")
    return eval(expression,{"__builtins__":{}},{})

In [61]:
def analyst_agent(question):
    # Lightweight deterministic tool router, illustrating tool binding and observations.
    ids=re.findall(r"\b\d{4}\b",question)
    observations={}
    if ids: observations["portfolio"]=portfolio_lookup(ids[0]).to_dict(orient="records")
    observations["market_context"]=market_search(question)
    if "project" in question.lower() and ids:
        rows=observations["portfolio"]
        if rows:
            observations["allocation_check"]={"equity_pct":rows[0]["equity_pct"],"bond_pct":rows[0]["bond_pct"]}
    return {"question":question,"tool_observations":observations,"note":"Synthetic demonstration; validate assumptions and consult qualified professionals for real financial decisions."}
analyst_agent("Compare customer 1001 portfolio against the market outlook and discuss risk considerations.")

{'question': 'Compare customer 1001 portfolio against the market outlook and discuss risk considerations.',
 'tool_observations': {'portfolio': [{'customer_id': 1001,
    'portfolio_balance': 132002.6,
    'equity_pct': 99.3,
    'bond_pct': 5.0,
    'risk_tolerance': 'low'}],
  'market_context': ['Q3 Market Outlook (synthetic demonstration only)\nThe sample outlook assumes elevated volatility, diversified exposure, and periodic portfolio rebalancing.',
   'Risk assessment should consider customer objectives, time horizon, liquidity needs, tax jurisdiction, and verified current holdings.']},
 'note': 'Synthetic demonstration; validate assumptions and consult qualified professionals for real financial decisions.'}

## Extension
Replace deterministic routing with a tool-calling LLM, define strict tool schemas, log tool inputs/outputs, add SQL read-only permissions, and require human approval before any external action.